In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_1.py ──
"""
Shared infrastructure for MLFP04 Exercise 1 — Clustering Zoo.

Contains: customer-feature loading & standardisation, metric helpers,
subsampling utilities, and output-directory management.

Technique-specific code (K-means elbow, linkage methods, DBSCAN epsilon
search, HDBSCAN, spectral Laplacian, AutoMLEngine) does NOT belong
here — it lives in the per-technique files under modules/mlfp04/solutions/ex_1/.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex1_clustering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared random state so every technique file is reproducible
RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════


def load_customers() -> tuple[pl.DataFrame, list[str]]:
    """Load the e-commerce customer dataset and return (df, numeric_feature_cols).

    The dataset (from MLFP03) is ~6K rows of Singapore e-commerce customers
    with recency, frequency, monetary, basket-size, and channel features.
    Clustering is unsupervised segmentation: no labels, just behaviour.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")
    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in ("customer_id",)
    ]
    return customers.drop_nulls(subset=feature_cols), feature_cols


def standardise(
    df: pl.DataFrame, feature_cols: list[str]
) -> tuple[np.ndarray, StandardScaler]:
    """Return (X_scaled, scaler). Zero mean, unit variance — mandatory for
    all distance-based clustering."""
    X, _, _ = to_sklearn_input(df, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — spectral / hierarchical are O(n^2) or worse
# ════════════════════════════════════════════════════════════════════════


def subsample(
    X: np.ndarray, n: int, seed: int = RANDOM_STATE
) -> tuple[np.ndarray, np.ndarray]:
    """Return (X_sub, idx) where idx are the original row indices chosen."""
    rng = np.random.default_rng(seed)
    n = min(n, X.shape[0])
    idx = rng.choice(X.shape[0], n, replace=False)
    return X[idx], idx


# ════════════════════════════════════════════════════════════════════════
# METRICS
# ════════════════════════════════════════════════════════════════════════


def score_partition(X: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    """Compute silhouette, Calinski-Harabasz, Davies-Bouldin.

    Points with label == -1 (DBSCAN noise) are excluded. Returns NaN
    fields if fewer than 2 valid clusters remain.
    """
    valid = labels != -1
    labs = labels[valid]
    data = X[valid]
    if data.shape[0] < 2 or len(set(labs.tolist())) < 2:
        return {
            "n_clusters": len(set(labs.tolist())),
            "silhouette": float("nan"),
            "calinski_harabasz": float("nan"),
            "davies_bouldin": float("nan"),
        }
    return {
        "n_clusters": len(set(labs.tolist())),
        "silhouette": float(silhouette_score(data, labs)),
        "calinski_harabasz": float(calinski_harabasz_score(data, labs)),
        "davies_bouldin": float(davies_bouldin_score(data, labs)),
    }


def agreement(labels_a: np.ndarray, labels_b: np.ndarray) -> dict[str, float]:
    """ARI and NMI between two label vectors, ignoring noise (-1)."""
    valid = (labels_a >= 0) & (labels_b >= 0)
    if valid.sum() < 2:
        return {"ari": float("nan"), "nmi": float("nan")}
    return {
        "ari": float(adjusted_rand_score(labels_a[valid], labels_b[valid])),
        "nmi": float(normalized_mutual_info_score(labels_a[valid], labels_b[valid])),
    }


def print_metric_row(name: str, m: dict[str, Any]) -> None:
    """One-line summary of a partition's metrics."""
    print(
        f"  {name:<14} K={m['n_clusters']:>3}  "
        f"sil={m['silhouette']:>7.4f}  "
        f"CH={m['calinski_harabasz']:>9.0f}  "
        f"DB={m['davies_bouldin']:>7.4f}"
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION OUTPUT PATH
# ════════════════════════════════════════════════════════════════════════


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 1.4: Spectral Clustering via the Graph Laplacian
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build an RBF affinity graph and the graph Laplacian
#   - Embed points via the smallest k eigenvectors of L
#   - Cluster the spectral embedding with K-means
#   - Recognise non-convex cluster shapes K-means cannot separate
#
# PREREQUISITES: 01_kmeans.py.
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — affinity, Laplacian, eigenvectors, embedding
#   2. Build — subsample + fit spectral for a few K values
#   3. Train — pick the best K
#   4. Visualise — silhouette vs K
#   5. Apply — SMRT Singapore train-line community detection
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import numpy as np
from dotenv import load_dotenv
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score

from kailash_ml import ModelVisualizer


load_dotenv()



## THEORY — Graph Laplacian Embedding


In [ ]:
# A_ij = exp(-||x_i - x_j||² / 2σ²)    (RBF affinity)
# D_ii = Σ_j A_ij                      (degree matrix)
# L = D - A                            (graph Laplacian)
# The smallest k eigenvectors of L embed the graph in R^k; points that
# are connected through dense paths land near each other there. Run
# K-means on the embedding. Price: O(n^3). Small-to-medium data only.



## TASK 2 — BUILD: subsample and fit spectral for K in {3,4,5}


In [ ]:
customers, feature_cols = load_customers()
X_scaled, _ = standardise(customers, feature_cols)
n_samples = X_scaled.shape[0]

# TODO: Subsample 2500 rows using the shared subsample() helper.
X_spec, idx_spec = ____
n_spec = X_spec.shape[0]

print("=" * 70)
print("  Spectral Clustering on Singapore E-commerce Customers")
print("=" * 70)
print(f"  Subsample: {n_spec:,} of {n_samples:,}")

K_CANDIDATES = [3, 4, 5]
spectral_results: dict[int, dict] = {}

print(f"\n  {'K':>3} {'Silhouette':>12} {'Time':>8}")
print("  " + "─" * 28)
for k in K_CANDIDATES:
    t0 = time.perf_counter()
    # TODO: Build a SpectralClustering model with n_clusters=k,
    # affinity="rbf", gamma=1.0, random_state=RANDOM_STATE,
    # assign_labels="kmeans". Fit_predict on X_spec.
    spec = ____
    labels = ____
    elapsed = time.perf_counter() - t0
    sil = silhouette_score(X_spec, labels)
    spectral_results[k] = {"labels": labels, "sil": sil, "time": elapsed}
    print(f"  {k:>3} {sil:>12.4f} {elapsed:>7.2f}s")



### Checkpoint 1


In [ ]:
assert len(spectral_results) == len(K_CANDIDATES), "Task 2: spectral sweep incomplete"
print("\n  [ok] Checkpoint 1 passed — spectral embeddings fitted\n")



## TASK 3 — TRAIN: Pick the best K and compare to K-means on the same X


In [ ]:
# TODO: Pick the K with the best silhouette from spectral_results.
best_k_spec, best_stats = ____
print(f"  Best spectral K: {best_k_spec}  (silhouette={best_stats['sil']:.4f})")

km_compare = KMeans(n_clusters=best_k_spec, random_state=RANDOM_STATE, n_init=10)
km_labels_sub = km_compare.fit_predict(X_spec)
km_sil = silhouette_score(X_spec, km_labels_sub)
print(f"    K-means silhouette (same subsample): {km_sil:.4f}")
print(f"    Δ (spectral − kmeans) = {best_stats['sil'] - km_sil:+.4f}")

spec_labels = best_stats["labels"]



### Checkpoint 2


In [ ]:
assert best_k_spec in K_CANDIDATES, "Task 3: best K selection invalid"
assert len(set(spec_labels.tolist())) == best_k_spec, "Task 3: label count mismatch"
print("\n  [ok] Checkpoint 2 passed — spectral best-K selected\n")



## TASK 4 — VISUALISE: silhouette vs K


In [ ]:
viz = ModelVisualizer()
fig = viz.training_history(
    {"Silhouette (spectral)": [spectral_results[k]["sil"] for k in K_CANDIDATES]},
    x_label="K",
)
fig.update_layout(title="Spectral Clustering: Silhouette vs K")
fig.write_html(str(out_path("04_spectral_silhouette.html")))
print(f"  Saved: {out_path('04_spectral_silhouette.html')}")



### Checkpoint 3


In [ ]:
assert out_path("04_spectral_silhouette.html").exists(), "Task 4: viz not saved"
print("\n  [ok] Checkpoint 3 passed — spectral visualisation rendered\n")



## TASK 5 — APPLY: SMRT Singapore Train-Line Community Detection


In [ ]:
# SCENARIO: SMRT's ~200 MRT/LRT stations form a natural graph with edge
# weights = inter-station rider flows. Spectral is the canonical
# community-detection method on graphs.
#
# BUSINESS IMPACT: ~S$29M / year (capacity matching + station-group
# advertising + incident rerouting) on ~S$850M rail revenue.

print("  APPLY — SMRT Train-Line Community Detection")
print("  ─────────────────────────────────────────────────────────────────")

# TODO: Compute sizes = np.bincount(spec_labels) and print each
# community's size + percentage of n_spec.
sizes = ____
for i, n in enumerate(sizes):
    print(f"    Community {i}: {n:>5,} customers ({n/n_spec:6.1%})")
print("    Estimated annual benefit: S$29M.")



### Checkpoint 4


In [ ]:
assert int(sizes.sum()) == n_spec, "Task 5: spectral partition size mismatch"
print("\n  [ok] Checkpoint 4 passed — spectral community partition valid\n")



## REFLECTION


[x] RBF affinity matrix and the graph Laplacian
  [x] Spectral embedding via the smallest k eigenvectors of L
  [x] K-means in the spectral embedding space
  [x] When spectral beats K-means (non-convex shapes, graph data)
  [x] Mapped to SMRT MRT community detection — ~S$29M / year

  Next: 05_evaluation_profiling.py — pick a winner.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

